# Imports

Brittany C. Haas and Melissa A. Hardy's jupyter notebook for automated collection of molecular descriptors and post-processing (i.e., Boltzmann average, min/max values, etc.).

**NOTE: Make sure to use the get_properties_environment file to set your conda environment.**

In [1]:
import os,re,sys,pickle,datetime,time,random,itertools,glob
import warnings
warnings.filterwarnings("ignore")
import openpyxl
import pandas as pd
from rdkit import Chem
import get_properties_functions as gp

D3 import failed


# Atom Inputs Dataframe

Portions of this section were adapted from code written Jordan P. Liles.

## Generate dataframe with atom numbers

### Use command line to prepare files

To create files: navigate to folder that contains all the log files you wish to analyze.

    module load openbabel
    obabel *.log -osdf -m  
    ls *.log > log_ids.txt
    cat *.sdf >> molecules.sdf

You will use the log_ids.txt and molecules.sdf files in the rest of 2.1.

### Define SMARTS substructure


Recommended to draw the common substructure (with general atoms) in Chemdraw and copy as SMILES (this will generate a SMARTS string)

More information about SMARTS and available characters here: https://www.daylight.com/dayhtml/doc/theory/theory.smarts.html


In [2]:
#this is the common smarts substructure for the molecules you will analyze
#you have to explicitly draw hydrogens into the SMARTS structure if you want to collect properties for hydrogen atoms
substructure = Chem.MolFromSmarts('[#6]S([#6])(~N[H])~N[H]')

### Generate preliminary dataframe

In [3]:
#generate a list of molecules using RDkit
all_compounds = Chem.SDMolSupplier('molecules.sdf', removeHs=False)
#molecules.sdf is generated with the instructions above
#it is a single sdf that contains the structures/atom numbers etc. for every molecule you will analyze

#uses RDKit to search for the substructure in each compound you will analyze
atoms = []
for molecule in all_compounds:
    if molecule is not None:
        submatch = molecule.GetSubstructMatches(substructure) #find substructure
        matchlist = list([item for sublist in submatch for item in sublist]) #list of zero-indexed atom numbers
        match_idx = [x+1 for x in matchlist] #this line changes from 0-indexed to 1-indexed (for Gaussian)
        atoms.append(match_idx) #append 1-indexed list to atoms (a list of lists)

#this loop extracts log names from log_ids and splits them to the desired format
filenames = open("log_ids.txt", "r") #generate this with instruction above
#it is a text file that contains the file name for every molecule you will analyze
list_of_filenames = [(line.strip()).split() for line in filenames] #list of the file names (each of which includes all conformers)
list_of_files = []
for filename in list_of_filenames:
    file = filename[0].split(".")
    list_of_files.append(file[0])
filenames.close()

#put the atom numbers for the substructure for each log file into a dataframe
prelim_df = pd.DataFrame(atoms)
index=list_of_files
prelim_df.insert(0,column='log_name',value=list_of_files)
display(prelim_df)

,log_name,0,1,2,3,4,5,6
0,3a_1,3,2,10,1,12,11,23
1,3b_1,3,2,11,1,13,12,24
2,3b_2,3,2,11,1,13,12,24
3,3c_1,3,2,14,1,16,15,24
4,3d_1,3,2,11,1,13,12,20
...,...,...,...,...,...,...,...,...
94,3u_5,23,24,26,25,39,27,43
95,3u_6,23,24,26,25,39,27,43
96,3u_7,23,24,26,27,43,25,39
97,3u_8,23,24,26,27,43,25,39


### Define column headers using GaussView

Using the preliminary dataframe displayed above, open one of your files and check the atom numbers. 

Assign labels to each column using the cell below. You will call these column headers when you select your properties. 

**User input required:**

In [4]:
atom_labels = {
                1: 'C1',
                2: 'S',
                3: 'C2',
                4: 'N1',
                5: 'H1',
                6: 'N2',
                7: 'H2'
                }

### Generate labeled dataframe

**NOTE: it is very important you assign these correctly otherwise the properties you collect will be for the wrong atoms and not produce meaningful correlations.** 

We recommend checking the numbering/headers for at least two different compounds. 

Numbering for different conformers of the same compounds will likely be the same (but may not be for some symmetrical groups).

In [5]:
#rename columns using the user input above
atom_map_df = prelim_df.rename(columns=atom_labels)
display(atom_map_df)

#you can use this to clean up the table if you have more atoms in your substructure than you want to collect descriptors for
#atom_map_df = atom_map_df.drop(columns= ['C4', 'C1'])
#display(atom_map_df.head())

df = atom_map_df #df is what properties will be appended to, this creates a copy so that you have the original preserved

,log_name,0,C1,S,C2,N1,H1,N2
0,3a_1,3,2,10,1,12,11,23
1,3b_1,3,2,11,1,13,12,24
2,3b_2,3,2,11,1,13,12,24
3,3c_1,3,2,14,1,16,15,24
4,3d_1,3,2,11,1,13,12,20
...,...,...,...,...,...,...,...,...
94,3u_5,23,24,26,25,39,27,43
95,3u_6,23,24,26,25,39,27,43
96,3u_7,23,24,26,27,43,25,39
97,3u_8,23,24,26,27,43,25,39


### Save atom map to Excel (if desired)

In [6]:
writer = pd.ExcelWriter('atom_map.xlsx')
atom_map_df.to_excel(writer)
writer.save()

## Import a manually-generated atom mapping dataframe

If you need to manually generate the atom mapping dataframe, check out the atom_map_sample.xlsx to make sure you have the desired format. 

In [ ]:
atom_map_df = pd.read_excel('Ac1_to_Ac5_sample_atom_map.xlsx','Sheet1',index_col=0,header=0,engine='openpyxl')
display(atom_map_df.head())

# Define Properties to Collect

## Available property functions and how to call them: 

In [7]:
#this box has functions to choose from
df = atom_map_df

#---------------GoodVibes Engergies---------------
#uses the GoodVibes 2021 Branch (Jupyter Notebook Compatible)
#calculates the quasi harmonic corrected G(T) and single point corrected G(T) as well as other thermodynamic properties
#inputs: dataframe, temperature
df = gp.get_goodvibes_e(df, 298.15)

#---------------Frontier Orbitals-----------------
#E(HOMO), E(LUMO), mu(chemical potential or negative of molecular electronegativity), eta(hardness/softness), omega(electrophilicity index)
df = gp.get_frontierorbs(df)

#---------------Polarizability--------------------
#Exact polarizability
df = gp.get_polarizability(df)

#---------------Dipole----------------------------
#Total dipole moment magnitude in Debye
df = gp.get_dipole(df)

#---------------Volume----------------------------
#Molar volume
#requires the Gaussian keyword = "volume" in the .com file
df = gp.get_volume(df)

#---------------SASA------------------------------
#Uses morfeus to calculat sovlent accessible surface area and the volume under the SASA
df = gp.get_SASA(df)

#---------------NBO-------------------------------
#natural charge from NBO
#requires the Gaussian keyword = "pop=nbo7" in the .com file
nbo_list = ['S', 'C1', 'C2', 'N1', 'H1', 'N2', 'H2']
df = gp.get_nbo(df, nbo_list)

#---------------NMR-------------------------------
#isotropic NMR shift
#requires the Gaussian keyword = "nmr=giao" in the .com file
nmr_list = ['S', 'C1', 'C2', 'N1', 'H1', 'N2', 'H2']
df = gp.get_nmr(df, nmr_list)

#---------------Distance--------------------------
#distance between 2 atoms
dist_list_of_lists = [['S', 'C1'], ['S', 'C2'], ['C1', 'C2'], ['S', 'N1'], ['S', 'N2'], ['N1', 'N2'], ['N1', 'H1'], ['N2', 'H2']]
df = gp.get_distance(df, dist_list_of_lists)

#---------------Dihedral--------------------------
#dihedral angle between 4 atoms
dihedral_list_of_lists = [['N2', 'S', 'N1', 'H1'], ['N1', 'S', 'N2', 'H2']]
df = gp.get_dihedral(df, dihedral_list_of_lists)

#---------------Angle-----------------------------
#angle between 3 atoms
angle_list_of_lists = [['C1', 'S', 'C2'], ['N1', 'S', 'N2']]
df = gp.get_angles(df, angle_list_of_lists)

#---------------Vbur Scan-------------------------
#uses morfeus to calculate the buried volume at a series of radii (including hydrogens)
#inputs: dataframe, list of atoms, start_radius, end_radius, and step_size
#if you only want a single radius, put the same value for start_radius and end_radius (keep step_size > 0)
vbur_list = ['S', 'C1', 'C2', 'N1', 'H1', 'N2', 'H2']
df = gp.get_vbur_scan(df, vbur_list, 2.0, 4.0, 0.5)

#---------------Sterimol morfeus------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values
#NOTE: this is much faster than the corresponding DBSTEP function (recommendation: use as default/if you don't need Sterimol2Vec)
sterimol_list_of_lists = [['S', 'C1'], ['S', 'C2']]
df = gp.get_sterimol_morfeus(df, sterimol_list_of_lists)

#---------------Buried Sterimol-------------------
#uses morfeus to calculate Sterimol L, B1, and B5 values within a given sphere of radius r_buried
#atoms outside the sphere + 0.5 vdW radius are deleted and the Sterimol vectors are calculated
#for more information: https://kjelljorner.github.io/morfeus/sterimol.html
#inputs: dataframe, list of atom pairs, r_buried
sterimol_list_of_lists = [['N', 'C3']]
df = gp.get_buried_sterimol(df, sterimol_list_of_lists, 5.5)

#---------------ChelpG----------------------------
#ChelpG ESP charge
#requires the Gaussian keyword = "pop=chelpg" in the .com file
a_list = ['S', 'C1', 'C2', 'N1', 'H1', 'N2', 'H2']
df = gp.get_chelpg(df, a_list)

#---------------Hirshfeld-------------------------
#Hirshfeld charge, CM5 charge, Hirshfeld atom dipole
#requires the Gaussian keyword = "pop=hirshfeld" in the .com file
a_list = ['S', 'C1', 'C2', 'N1', 'H1', 'N2', 'H2']
df = gp.get_hirshfeld(df, a_list)

pd.options.display.max_columns = None
display(df)



   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational scale factor 1.0 for PBE1PBE/def2SVP level of theory

   Using vibrational sc

,log_name,0,C1,S,C2,N1,H1,N2,E_spc (Hartree),ZPE(Hartree),H_spc(Hartree),T*S,T*qh_S,G(T)_spc(Hartree),qh_G(T)_spc(Hartree),T,HOMO,LUMO,μ,η,ω,polar_iso(au),polar_aniso(au),dipole(Debye),volume(Bohr_radius³/mol),SASA_surface_area(Å²),SASA_volume(Å³),SASA_sphericity,NBO_charge_S,NBO_charge_C1,NBO_charge_C2,NBO_charge_N1,NBO_charge_H1,NBO_charge_N2,NBO_charge_H2,NMR_shift_S,NMR_shift_C1,NMR_shift_C2,NMR_shift_N1,NMR_shift_H1,NMR_shift_N2,NMR_shift_H2,angle_C1_S_C2(°),angle_N1_S_N2(°),%Vbur_S_2.0Å,%Vbur_C1_2.0Å,%Vbur_C2_2.0Å,%Vbur_N1_2.0Å,%Vbur_H1_2.0Å,%Vbur_N2_2.0Å,%Vbur_H2_2.0Å,%Vbur_S_2.5Å,%Vbur_C1_2.5Å,%Vbur_C2_2.5Å,%Vbur_N1_2.5Å,%Vbur_H1_2.5Å,%Vbur_N2_2.5Å,%Vbur_H2_2.5Å,%Vbur_S_3.0Å,%Vbur_C1_3.0Å,%Vbur_C2_3.0Å,%Vbur_N1_3.0Å,%Vbur_H1_3.0Å,%Vbur_N2_3.0Å,%Vbur_H2_3.0Å,%Vbur_S_3.5Å,%Vbur_C1_3.5Å,%Vbur_C2_3.5Å,%Vbur_N1_3.5Å,%Vbur_H1_3.5Å,%Vbur_N2_3.5Å,%Vbur_H2_3.5Å,%Vbur_S_4.0Å,%Vbur_C1_4.0Å,%Vbur_C2_4.0Å,%Vbur_N1_4.0Å,%Vbur_H1_4.0Å,%Vbur_N2_4.0Å,%Vbur_H2_4.0Å,Sterimol_L_S_C1(Å)_morfeus,Sterimol_B1_S_C1(Å)_morfeus,Sterimol_B5_S_C1(Å)_morfeus,Sterimol_L_S_C2(Å)_morfeus,Sterimol_B1_S_C2(Å)_morfeus,Sterimol_B5_S_C2(Å)_morfeus,ChelpG_charge_S,ChelpG_charge_C1,ChelpG_charge_C2,ChelpG_charge_N1,ChelpG_charge_H1,ChelpG_charge_N2,ChelpG_charge_H2,Hirsh_charge_S,Hirsh_CM5_charge_S,Hirsh_atom_dipole_S,Hirsh_charge_C1,Hirsh_CM5_charge_C1,Hirsh_atom_dipole_C1,Hirsh_charge_C2,Hirsh_CM5_charge_C2,Hirsh_atom_dipole_C2,Hirsh_charge_N1,Hirsh_CM5_charge_N1,Hirsh_atom_dipole_N1,Hirsh_charge_H1,Hirsh_CM5_charge_H1,Hirsh_atom_dipole_H1,Hirsh_charge_N2,Hirsh_CM5_charge_N2,Hirsh_atom_dipole_N2,Hirsh_charge_H2,Hirsh_CM5_charge_H2,Hirsh_atom_dipole_H2
0,3a_1,3,2,10,1,12,11,23,-819.231702,0.188653,-819.030437,0.051013,0.049281,-819.081450,-819.079718,298.15,-0.24369,-0.03383,-0.138760,0.20986,0.04587,130.797,71.0290,2.9060,1366.145,356.651035,527.356068,0.885068,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,36.404,55.215,92.920325,98.921746,70.273760,58.212810,71.616736,57.660770,no data,73.547988,93.612450,58.910665,48.616356,59.641555,46.996679,no data,56.764005,82.981973,50.628538,41.231190,51.006593,39.692901,no data,47.357203,67.218948,44.642774,36.553648,44.946771,35.345229,no data,40.462650,50.995050,38.652939,32.964279,39.228245,31.727158,no data,5.553157,2.376853,7.671678,4.966082,1.727057,8.310670,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data
1,3b_1,3,2,11,1,13,12,24,-894.388236,0.193526,-894.181231,0.052332,0.051056,-894.233563,-894.232287,298.15,-0.24631,-0.02802,-0.137165,0.21829,0.04309,137.384,81.4411,5.5650,1507.162,370.380363,551.889338,0.878491,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,30.620,69.231,93.013946,99.105759,68.908187,65.202092,69.805656,58.109504,no data,73.966337,93.825693,57.167274,55.850371,58.396276,47.994531,no data,57.240763,82.898167,49.244823,47.181354,50.046558,40.388483,no data,47.671100,67.078597,43.626537,40.749045,44.134947,35.761042,no data,40.502995,50.869747,38.088108,35.678651,38.347635,32.364146,no data,5.999813,2.195783,8.692337,4.738285,1.719411,9.360998,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data
2,3b_2,3,2,11,1,13,12,24,-894.395872,0.194251,-894.188401,0.051424,0.050441,-894.239825,-894.238842,298.15,-0.24001,-0.02524,-0.132625,0.21477,0.04095,136.779,80.7546,3.4788,1583.341,371.010427,551.958089,0.877072,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,no data,36.439,55.435,92.852531,98.905604,70.202738,58.154700,71.723270,57.751162,no data,73.510548,93.628728,58.801

In [ ]:
df.to_csv('../results/old_individual_properties.csv')

## Copy and modify available property functions above to customize

We recommend copying the entire cell above. You will need to change the atom number lists to match your desired column headers and delete (or comment out) any properites you don't want to collect.

In [ ]:
#this box has functions to choose from
df = atom_map_df

#---------------GoodVibes Engergies---------------
#uses the GoodVibes 2021 Branch (Jupyter Notebook Compatible)
#calculates the quasi harmonic corrected G(T) and single point corrected G(T) as well as other thermodynamic properties
#inputs: dataframe, temperature
df = gp.get_goodvibes_e(df, 298.15)

#---------------Frontier Orbitals-----------------
#E(HOMO), E(LUMO), mu(chemical potential or negative of molecular electronegativity), eta(hardness/softness), omega(electrophilicity index)
df = gp.get_frontierorbs(df)

#---------------Vbur Scan-------------------------
#uses morfeus to calculate the buried volume at a series of radii (including hydrogens)
#inputs: dataframe, list of atoms, start_radius, end_radius, and step_size
#if you only want a single radius, put the same value for start_radius and end_radius (keep step_size > 0)
vbur_list = ["C1", "C4"]
df = gp.get_vbur_scan(df, vbur_list, 2, 4, 0.5)


#---------------Sterimol DBSTEP-------------------
#uses DBSTEP to calculate Sterimol L, B1, and B5 values
#default grid point spacing (0.05 Angstrom) is used (can use custom spacing or vdw radii in the get_properties_functions script)
#more info here: https://github.com/patonlab/DBSTEP
#NOTE: this takes longer than the morfeus function (recommendation: only use this if you need Sterimol2Vec)
sterimol_list_of_lists = [["O3", "H5"]]
df = gp.get_sterimol_dbstep(df, sterimol_list_of_lists)

#---------------Sterimol2Vec----------------------
#uses DBSTEP to calculate Sterimol Bmin and Bmax values at intervals from 0 to end_radius, with a given step_size
#default grid point spacing (0.05 Angstrom) is used (can use custom spacing or vdw radii in the get_properties_functions script)
#more info here: https://github.com/patonlab/DBSTEP
#inputs: dataframe, list of atom pairs, end_radius, and step_size
sterimol2vec_list_of_lists = [["C1", "C4"]]
df = gp.get_sterimol2vec(df, sterimol2vec_list_of_lists, 1, 1.0)

#---------------Pyramidalization------------------
#uses morfeus to calculate pyramidalization based on the 3 atoms in closest proximity to the defined atom
#collects values based on two definitions of pyramidalization
#details on these values can be found here: https://kjelljorner.github.io/morfeus/pyramidalization.html
pyr_list = ["C1", "C4"]
df = gp.get_pyramidalization(df, pyr_list)

#---------------Plane Angle-----------------------
#plane angle between 2 planes (each defined by 3 atoms)
planeangle_list_of_lists = [["O2", "C1", "O3", "H5", "C1", "C4"], ["O2", "C1", "O3", "H5", "C1", "C4"]]
df = gp.get_planeangle(df, planeangle_list_of_lists)

#---------------Time----------------------------------
#returns the total CPU time and total Wall time
#if used in summary df, will give the average (not Boltzmann average) in the Boltzmann average column
df = gp.get_time(df)

#---------------IR--------------------------------
#CAUTION: CANNOT ACCURATELY IDENTIFY ATOM STRETCHES IN SOME CASES (struggles if there is more than one stretch in the defined range)
#IR frequencies and intensities in a specific range (for specific atoms)
#requires the Gaussian keyword = "freq=noraman" in the .com file
#inputs: dataframe, atom1, atom2, frequency_min, frequency_max, intensity_min, intensity_max, threshold
#if you want to collect multiple IR frequencies, you will need to copy/paste this function for each stretch
#we recommend a threshold of 0.0 (may have to adjust)
df = gp.get_IR(df, "C1", "O2", 1700, 1900, 100, 800, 0.0)


pd.options.display.max_columns = None
display(df)

## Save collected properties to Excel

Helpful to save here in case the Notebook crashes or if you want to add more properties before post-processsing. Can be read in at 5.1.1.

In [ ]:
writer = pd.ExcelWriter('All_Conformer_Properties_example.xlsx')
df.to_excel(writer)
writer.save()

# Post-processing

## User input for data processing

In [ ]:
#for numerically named compounds, prefix is any text common to all BEFORE the number and suffix is common to all AFTER the number
#this is a template for our files that are all named "AcXXX_clust-X.log" or "AcXXX_conf-X.log"
prefix = ""
suffix = "_"

#columns that provide atom mapping information are dropped
atom_columns_to_drop = ['C1','N', 'C1_prime','C2_prime', 'C3', 'C2']

#title of the column for the energy you want to use for boltzmann averaging and lowest E conformer determination
energy_col_header = "G(T)_spc(Hartree)"


energy_cutoff = 4.2 #specify energy cutoff in kcal/mol to remove conformers above this value before post-processing
verbose = False #set to true if you'd like to see info on the nunmber of conformers removed for each compound

### Option to import an Excel sheet if you're using properties or energies collected outside of this notebook

If you would like to use post-processing functionality (i.e. Boltzmann averaging, lowest E conformers, etc.) you can read in a dataframe with properties (e.g. QikProp properties) or energies (e.g. if you don't/can't run linked jobs) collected outside of this notebook. 

Check out the dataframe_sample.xlsx to make sure you have the desired format. 

In [ ]:
df = pd.read_csv('/Users/jameshoward/Documents/Programming/GetProperties/tests/conformer_specific_properties_new.csv', header=0)
display(df)

## Generating a list of compounds that have conformational ensembles

**ONLY RUN THE AUTOMATED OR THE MANUAL CELL, NOT BOTH**

**AUTOMATED:** if your compounds are named consistenly, this section generates your compound list based on the similar naming structure

In [ ]:
#this is a template for our files that are all named "AcXXX_clust-X.log"
from pathlib import Path
compound_list = []

for index, row in df.iterrows():
    log_file = Path(row['log_name']).stem #read file name from df
    compound = log_file.split(str(suffix))[0] #splits again to get "XXX" (entry 1) (and we don't use the empty string "" (entry 0))
    compound_list.append(compound)

compound_list = list(set(compound_list)) #removes duplicate stuctures that result from having conformers of each
compound_list.sort() #reorders numerically (not sure if it reorders alphabetically)
print(compound_list)

#this should generate a list that looks like this: ['24', '27', '34', '48']

**MANUAL:** if your comment naming scheme is not consistent or you have trouble with the template above, you can manually define your compound list

In [ ]:
compound_list = [1, 2, 3, 4, 5]

## Post-processing to get properties for each compound

In [ ]:
all_df_master = pd.DataFrame(columns=[])
properties_df_master = pd.DataFrame(columns=[])
import numpy as np

for compound in compound_list:
    #defines the common start to all files using the input above
    substring = str(prefix) + str(compound) + str(suffix)

    #makes a data frame for one compound at a time for post-processing
    valuesdf = df[df["log_name"].str.startswith(substring)]
    valuesdf = valuesdf.drop(columns = atom_columns_to_drop)
    valuesdf = valuesdf.reset_index(drop = True)  #you must re-index otherwise the 2nd, 3rd, etc. compounds fail

    #define columns that won't be included in summary properties or are treated differently because they don't make sense to Boltzmann average
    non_boltz_columns = ["G(Hartree)","∆G(Hartree)","∆G(kcal/mol)", "e^(-∆G/RT)","Mole Fraction"] #don't boltzman average columns containing these strings in the column label
    reg_avg_columns = ['CPU_time_total(hours)', 'Wall_time_total(hours)'] #don't boltzmann average these either, we average them in case that is helpful
    gv_extra_columns = ['E_spc (Hartree)', 'H_spc(Hartree)', 'T', 'T*S', 'T*qh_S', 'ZPE(Hartree)', 'qh_G(T)_spc(Hartree)', "G(T)_spc(Hartree)"]
    gv_extra_columns.remove(str(energy_col_header))

    #calculate the summary properties based on all conformers (Boltzmann Average, Minimum, Maximum, Boltzmann Weighted Std)
    valuesdf["∆G(Hartree)"] = valuesdf[energy_col_header] - valuesdf[energy_col_header].min()
    valuesdf["∆G(kcal/mol)"] = valuesdf["∆G(Hartree)"] * 627.5
    valuesdf["e^(-∆G/RT)"] = np.exp((valuesdf["∆G(kcal/mol)"] * -1000) / (1.987204 * 298.15)) #R is in cal/(K*mol)
    valuesdf["Mole Fraction"] = valuesdf["e^(-∆G/RT)"] / valuesdf["e^(-∆G/RT)"].sum()
    initial = len(valuesdf.index)
    if verbose:
        print(prefix + str(compound))
        #display(valuesdf)
        print("Total number of conformers = ", initial)
    valuesdf.drop(valuesdf[valuesdf["∆G(kcal/mol)"] >= energy_cutoff].index, inplace=True) #E cutoff applied here
    valuesdf = valuesdf.reset_index(drop = True) #resetting indexes
    final = len(valuesdf.index)
    if verbose:
        print("Number of conformers above ", energy_cutoff, " kcal/mol: ", initial-final)
    values_boltz_row = []
    values_min_row = []
    values_max_row = []
    values_boltz_stdev_row =[]
    values_range_row = []
    values_exclude_columns = []

    for column in valuesdf:
        if "log_name" in column:
            values_boltz_row.append("Boltzmann Averages")
            values_min_row.append("Ensemble Minimum")
            values_max_row.append("Ensemble Maximum")
            values_boltz_stdev_row.append("Boltzmann Standard Deviation")
            values_range_row.append("Ensemble Range")
            values_exclude_columns.append(column) #used later to build final dataframe
        elif any(phrase in column for phrase in non_boltz_columns) or any(phrase in column for phrase in gv_extra_columns):
            values_boltz_row.append("")
            values_min_row.append("")
            values_max_row.append("")
            values_boltz_stdev_row.append("")
            values_range_row.append("")
        elif any(phrase in column for phrase in reg_avg_columns):
            values_boltz_row.append(valuesdf[column].mean()) #intended to print the average CPU/wall time in the boltz column
            values_min_row.append("")
            values_max_row.append("")
            values_boltz_stdev_row.append("")
            values_range_row.append("")
        else:
            valuesdf[column] = pd.to_numeric(valuesdf[column]) #to hopefully solve the error that sometimes occurs where the float(Mole Fraction) cannot be mulitplied by the string(property)
            values_boltz_row.append((valuesdf[column] * valuesdf["Mole Fraction"]).sum())
            values_min_row.append(valuesdf[column].min())
            values_max_row.append(valuesdf[column].max())
            values_range_row.append(valuesdf[column].max() - valuesdf[column].min())


            # this section generates the weighted std deviation (weighted by mole fraction)
            # formula: https://www.statology.org/weighted-standard-deviation-excel/

            boltz = (valuesdf[column] * valuesdf["Mole Fraction"]).sum() #number
            delta_values_sq = []

            #makes a list of the "deviation" for each conformer
            for index, row in valuesdf.iterrows():
                value = row[column]
                delta_value_sq = (value - boltz)**2
                delta_values_sq.append(delta_value_sq)

            #w is list of weights (i.e. mole fractions)
            w = list(valuesdf["Mole Fraction"])
            wstdev = np.sqrt( (np.average(delta_values_sq, weights=w)) / (((len(w)-1)/len(w))*np.sum(w)) )
            if len(w) == 1: #if there is only one conformer in the ensemble, set the weighted standard deviation to 0
                wstdev = 0
            #np.average(delta_values_sq, weights=w) generates sum of each (delta_value_sq * mole fraction)

            values_boltz_stdev_row.append(wstdev)


    valuesdf.loc[len(valuesdf)] = values_boltz_row
    valuesdf.loc[len(valuesdf)] = values_boltz_stdev_row
    valuesdf.loc[len(valuesdf)] = values_min_row
    valuesdf.loc[len(valuesdf)] = values_max_row
    valuesdf.loc[len(valuesdf)] = values_range_row

    #final output format is built here:
    explicit_order_front_columns = ["log_name", energy_col_header,"∆G(Hartree)","∆G(kcal/mol)","e^(-∆G/RT)","Mole Fraction"]

    #reorders the dataframe using front columns defined above
    valuesdf = valuesdf[explicit_order_front_columns + [col for col in valuesdf.columns if col not in explicit_order_front_columns and col not in values_exclude_columns]]

    #determine the index of the lowest energy conformer
    low_e_index = valuesdf[valuesdf["∆G(Hartree)"] == 0].index.tolist()

    #copy the row to a new_row with the name of the log changed to Lowest E Conformer
    new_row = valuesdf.loc[low_e_index[0]]
    new_row['log_name'] = "Lowest E Conformer"
    valuesdf =  valuesdf.append(new_row, ignore_index=True)

#------------------------------EDIT THIS SECTION IF YOU WANT A SPECIFIC CONFORMER----------------------------------
#    #if you want all properties for a conformer with a particular property (i.e. all properties for the Vbur_min conformer)
#    #this template can be adjusted for min/max/etc.
#
#    #find the index for the min or max column:
#    ensemble_min_index = valuesdf[valuesdf["log_name"] == "Ensemble Minimum"].index.tolist()
#
#    #find the min or max value of the property (based on index above)
#    #saves the value in a list (min_value) with one entry (this is why we call min_value[0])
#    min_value = valuesdf.loc[ensemble_min_index, "%Vbur_C4_3.0Å"].tolist()
#    vbur_min_index = valuesdf[valuesdf["%Vbur_C4_3.0Å"] == min_value[0]].index.tolist()
#
#    #copy the row to a new_row with the name of the log changed to Property_min_conformer
#    new_row = valuesdf.loc[vbur_min_index[0]]
#    new_row['log_name'] = "%Vbur_C4_3.0Å_min_Conformer"
#    valuesdf =  valuesdf.append(new_row, ignore_index=True)
#--------------------------------------------------------------------------------------------------------------------

    #appends the frame to the master output
    all_df_master = pd.concat([all_df_master, valuesdf])

    #drop all the individual conformers
    dropindex = valuesdf[valuesdf["log_name"].str.startswith(substring)].index
    valuesdf = valuesdf.drop(dropindex)
    valuesdf = valuesdf.reset_index(drop = True)

    #display(valuesdf)

    #drop the columns created to determine the mole fraction and some that
    valuesdf = valuesdf.drop(columns = explicit_order_front_columns)
    try:
        valuesdf = valuesdf.drop(columns = gv_extra_columns)
    except:
        pass
    try:
        valuesdf = valuesdf.drop(columns = reg_avg_columns)
    except:
        pass

#---------------------THIS MAY NEED TO CHANGE DEPENDING ON HOW YOU LABEL YOUR COMPOUNDS------------------------------
    compound_name = prefix + str(compound)
#--------------------------------------------------------------------------------------------------------------------

    properties_df = pd.DataFrame({'Compound_Name': [compound_name]})

    #builds a dataframe (for each compound) by adding summary properties as new columns
    for (columnName, columnData) in valuesdf.iteritems():
        #the indexes need to match the values dataframe - display it to double check if you need to make changes
        #(uncomment the display(valuesdf) in row 124 of this cell)
        properties_df[str(columnName) + "_Boltz"] = [columnData.values[0]]
        properties_df[str(columnName) + "_Boltz_stdev"] = [columnData.values[1]]
        properties_df[str(columnName) + "_min"] = [columnData.values[2]]
        properties_df[str(columnName) + "_max"] = [columnData.values[3]]
        properties_df[str(columnName) + "_range"] = [columnData.values[4]]
        properties_df[str(columnName) + "_low_E"] = [columnData.values[5]]

        #if you're collecting properties for a specific conformer, add these here (note the index)
        #example:
        #properties_df[str(columnName) + "_V_bur_min"] = [columnData.values[6]]

        #if you only want a table with Boltz, you can comment out the other summary properties to generate a Boltz spreadsheet
        #of if you don't want to collect range, etc.
    #concatenates the individual acid properties df into the master properties df
    properties_df_master = pd.concat([properties_df_master, properties_df], axis = 0)

all_df_master = all_df_master.reset_index(drop = True)
properties_df_master = properties_df_master.reset_index(drop = True)


### Peek at your new dataframes

In [16]:
display(properties_df_master)
properties_df_master.to_csv('/Users/jameshoward/Documents/Programming/GetProperties/tests/summary_properties_old.csv')

,Compound_Name,Unnamed: 0_Boltz,Unnamed: 0_Boltz_stdev,Unnamed: 0_min,Unnamed: 0_max,Unnamed: 0_range,Unnamed: 0_low_E,HOMO_Boltz,HOMO_Boltz_stdev,HOMO_min,...,Hirsh_CM5_charge_C1_min,Hirsh_CM5_charge_C1_max,Hirsh_CM5_charge_C1_range,Hirsh_CM5_charge_C1_low_E,Hirsh_atom_dipole_C1_Boltz,Hirsh_atom_dipole_C1_Boltz_stdev,Hirsh_atom_dipole_C1_min,Hirsh_atom_dipole_C1_max,Hirsh_atom_dipole_C1_range,Hirsh_atom_dipole_C1_low_E
0,10012430,623.888094,108.192878,360.0,682.0,322.0,682.0,-0.291670,0.000130,-0.29183,...,0.257396,0.259142,0.001746,0.259142,0.263010,0.000141,0.261670,0.263040,0.001369,0.263011
1,10012462,2365.711964,10.315389,2181.0,2366.0,185.0,2366.0,-0.304684,0.000153,-0.30742,...,0.223597,0.229378,0.005781,0.229378,0.248565,0.000389,0.241605,0.248576,0.006970,0.248576
2,10012635,93.945771,141.292749,56.0,585.0,529.0,56.0,-0.313460,0.000716,-0.31547,...,0.025805,0.031178,0.005373,0.026372,0.049750,0.000673,0.047292,0.049929,0.002637,0.049929
3,10034763,699.000000,0.000000,699.0,699.0,0.0,699.0,-0.332330,0.000000,-0.33233,...,0.134168,0.134168,0.000000,0.134168,0.142325,0.000000,0.142325,0.142325,0.000000,0.142325
4,10034794,1050.285206,385.449989,387.0,2530.0,2143.0,939.0,-0.278048,0.002644,-0.28367,...,0.039607,0.043347,0.003740,0.042458,0.062395,0.001775,0.059959,0.073909,0.013950,0.062100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
379,12633569,2509.984768,214.497412,2268.0,2809.0,541.0,2513.0,-0.319253,0.000483,-0.31969,...,0.241068,0.243131,0.002063,0.241151,0.258500,0.001136,0.257860,0.260017,0.002158,0.257861
380,12640226,1178.593335,116.485026,917.0,1342.0,425.0,1263.0,-0.295339,0.000106,-0.29563,...,0.090729,0.090847,0.000118,0.090847,0.100855,0.000264,0.100779,0.101572,0.000793,0.100784
381,12648543,1239.218313,368.387007,735.0,1903.0,1168.0,1609.0,-0.338903,0.003025,-0.34203,...,0.056829,0.058064,0.001235,0.057936,0.073173,0.000234,0.072938,0.073759,0.000821,0.073000
382,13739739,1062.613203,867.086912,393.0,2238.0,1845.0,568.0,-0.331781,0.001856,-0.33547,...,0.038558,0.041277,0.002719,0.039870,0.056162,0.000719,0.055613,0.057804,0.002191,0.055762


### Save to Microsoft Excelᵀᴹ 

In [ ]:
all_df_master.to_excel('All_Conformer_and_Summary_Properties_example.xlsx', index = False)
properties_df_master.to_excel('Summary_Properties_example.xlsx', index = False)